In [51]:
import pandas as pd
import os
import numpy as np
import calendar

PARAMS      = ['RAINFALL_24H_MM', 'TEMPERATURE_AVG_C', 'TEMP_24H_TN_C', 'TEMP_24H_TX_C']
BASELINES   = ['1991', '1981']
METDAT_COLS = ['NAME', 'WMO_ID', 'DATA_TIMESTAMP', 'CURRENT_LATITUDE', 'CURRENT_LONGITUDE', 'PROVINSI', 'KABUPATEN', 'ELEVATION']

WORKING_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_ROOT     = os.path.join(WORKING_DIR, 'data')
LONG_DIR      = os.path.join(DATA_ROOT, '05.Long_Format_Dataset')

# 1. Persiapan Data: Filter data harian suhu rata-rata dari hasil homogenisasi dengan baseline 1991
def get_anomali_dataset(path):
    df_path    = os.path.join(path, '04.DATA_HOMO_DB.csv')
    df_homo    = pd.read_csv(df_path)
    df_anomali = df_homo[(df_homo['parameter'] == 'TEMPERATURE_AVG_C') &(df_homo['source']    == 'homogenisasi') &(df_homo['baseline']  == 1991)].copy()
    df_anomali['time'] = pd.to_datetime(df_anomali['time'])
    return df_anomali

def get_valid_agg_monthly(df):
    # Validasi input
    assert pd.api.types.is_datetime64_any_dtype(df['time']), "'time' must be datetime"
    df_monthly = df.groupby(['wmo_id', pd.Grouper(key='time', freq='ME')]).agg(
        parameter=('parameter', 'first'),
        source=('source', 'first'),
        baseline=('baseline', 'first'),
        name=('name', 'first'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first'),
        province=('provinsi', 'first'),
        regency=('kabupaten', 'first'),
        elevation=('elevasi', 'first'),
        value=('value', 'mean'),
        days_present=('value', 'count')
    ).reset_index()
    # Hitung kelengkapan
    df_monthly['days_in_month']     = df_monthly['time'].dt.daysinmonth
    df_monthly['data_completeness'] = (df_monthly['days_present'] / df_monthly['days_in_month']) * 100
    # Masking jika <80%
    mask_invalid = df_monthly['data_completeness'] < 80
    df_monthly.loc[mask_invalid, 'value'] = pd.NA  # atau np.nan
    df_monthly['is_valid_for_normal'] = ~mask_invalid
    # Tambahkan year/month untuk analisis
    df_monthly['month'] = df_monthly['time'].dt.month
    df_monthly['year']  = df_monthly['time'].dt.year
    return df_monthly


def get_dataNormal(df_monthly, start_year=1991, end_year=2020):
    clim_period_data   = df_monthly[(df_monthly['year'] >= start_year) & (df_monthly['year'] <= end_year)].copy()
    data_normal        = clim_period_data[clim_period_data['is_valid_for_normal']]
    data_normal        = data_normal[data_normal['is_valid_for_normal'] == True]
    #persetse data perbuanan untuk setiap stasiun
    data_count_per_station = data_normal.groupby(['wmo_id', 'month'])['time'].nunique().reset_index()
    data_count_per_station['valid_for_normal'] = data_count_per_station['time'] >= 24  # Minimal 24 tahun data valid
    valid_stations_per_month = data_count_per_station[data_count_per_station['valid_for_normal']==True]
    valid_data_for_normal = pd.merge(data_normal, valid_stations_per_month[['wmo_id', 'month']], on=['wmo_id', 'month'], how='inner')
    #buat NORMAL berdasarkan data valid tersebut buat NaN untuk stasiun yang tidak valid
    dataNormal = valid_data_for_normal.groupby(['wmo_id', 'month'])['value'].mean().reset_index()
    dataNormal.rename(columns={'value': 'normal'}, inplace=True)
    # print jumlah stasiun pada dataNormal per bulan
    print(dataNormal.groupby('month')['wmo_id'].nunique())
    return dataNormal

df                  = get_anomali_dataset(LONG_DIR).copy()
df_monthly          = get_valid_agg_monthly(df).copy()
df_monthly.to_csv('check.csv')

In [50]:
start_year = 1991
end_year   = 2020
clim_period_data       = df_monthly[(df_monthly['year'] >= start_year) & (df_monthly['year'] <= end_year)].copy()
data_normal            = clim_period_data[clim_period_data['is_valid_for_normal']]
data_count_per_station = data_normal.groupby(['wmo_id', 'month'])['time'].nunique().reset_index()
data_count_per_station['valid_for_normal'] = data_count_per_station['time'] >= 24  # Minimal 24 tahun data valid
valid_stations_per_month = data_count_per_station[data_count_per_station['valid_for_normal']==True]
valid_data_for_normal = pd.merge(data_normal, valid_stations_per_month[['wmo_id', 'month']], on=['wmo_id', 'month'], how='inner')
dataNormal = valid_data_for_normal.groupby(['wmo_id', 'month'])['value'].mean().reset_index()
dataNormal.rename(columns={'value': 'normal'}, inplace=True)
print(dataNormal.groupby('month')['wmo_id'].nunique())

month
1     118
2     118
3     118
4     118
5     118
6     118
7     118
8     118
9     118
10    118
11    118
12    118
Name: wmo_id, dtype: int64


In [34]:
clim_period_data[clim_period_data['data_completeness'] < 80]

,wmo_id,time,parameter,source,baseline,name,latitude,longitude,province,regency,elevation,value,days_present,days_in_month,data_completeness,is_valid_for_normal,month,year
1160,96011,2017-05-31,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Sultan Iskandar Muda,5.52244,95.41700,Nanggroe Aceh Darussalam,Kab. Aceh Besar,20.0,NaN,22,31,70.967742,False,5,2017
1161,96011,2017-06-30,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Sultan Iskandar Muda,5.52244,95.41700,Nanggroe Aceh Darussalam,Kab. Aceh Besar,20.0,NaN,22,30,73.333333,False,6,2017
1162,96011,2017-07-31,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Sultan Iskandar Muda,5.52244,95.41700,Nanggroe Aceh Darussalam,Kab. Aceh Besar,20.0,NaN,18,31,58.064516,False,7,2017
1164,96011,2017-09-30,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Sultan Iskandar Muda,5.52244,95.41700,Nanggroe Aceh Darussalam,Kab. Aceh Besar,20.0,NaN,22,30,73.333333,False,9,2017
1165,96011,2017-10-31,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Sultan Iskandar Muda,5.52244,95.41700,Nanggroe Aceh Darussalam,Kab. Aceh Besar,20.0,NaN,23,31,74.193548,False,10,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48620,97900,2019-05-31,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Mathilda Batlayeri,-7.98000,131.30000,Maluku,Kab. Kep. Tanimbar,24.0,NaN,2,31,6.451613,False,5,2019
48624,97900,2019-09-30,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Mathilda Batlayeri,-7.98000,131.30000,Maluku,Kab. Kep. Tanimbar,24.0,NaN,1,30,3.333333,False,9,2019
48990,97980,2015-09-30,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Mopah,-8.52019,140.41568,Papua,Kab. Merauke,0.0,NaN,11,30,36.666667,False,9,2015
48993,97980,2015-12-31,TEMPERATURE_AVG_C,homogenisasi,1991,Stasiun Meteorologi Mopah,-8.52019,140.41568,Papua,Kab. Merauke,0.0,NaN,24,31,77.419355,False,12,2015


In [ ]:
df_monthly['month'] = df_monthly['time'].dt.month
df_monthly.to_csv(os.path.join(LONG_DIR, '06.TEMPERATURE_MONTHLY_DB.csv'))

# Dapatkan normal (pastikan kolomnya bernama 'normal')
# dataNormal          = get_dataNormal(df_monthly, start_year=1991, end_year=2020)
# dataNormal          = dataNormal.dropna(subset=['normal'])  # Hapus NaN di kolom normal
# dataNormal.to_csv(os.path.join(LONG_DIR, '06.TEMPERATURE_ANOMALI_NORMAL_DB.csv'), index=False)
dataNormal = pd.read_csv(os.path.join(LONG_DIR, '06.TEMPERATURE_ANOMALI_NORMAL_DB.csv'))

#Hitung anomali bulanan
dataAnomali = pd.DataFrame()
for month in range(1, 13):
    normal_bulanan = dataNormal[dataNormal['month'] == month]
    wmoids         = normal_bulanan['wmo_id'].unique()
    df_bulanan     = df_monthly[(df_monthly['month'] == month) & (df_monthly['wmo_id'].isin(wmoids))].copy()
    df_merge            = pd.merge(df_bulanan, normal_bulanan[['wmo_id', 'normal']], on='wmo_id', how='left')
    df_merge['anomali'] = df_merge['value'] - df_merge['normal']
    df_merge['suhu_bulan_sebelum']    = df_merge['value'].shift(1)
    df_merge['selisih_suhu']          = (df_merge['value'] - df_merge['suhu_bulan_sebelum'])
    dataAnomali = pd.concat([dataAnomali, df_merge], ignore_index=True)

# 9. Sortir dan tambahkan informasi tambahan
dataAnomali                             = dataAnomali.sort_values(['wmo_id', 'time']).reset_index(drop=True)
dataAnomali['anomali_diff']             = dataAnomali.groupby('wmo_id')['anomali'].diff()
dataAnomali['year']                     = dataAnomali['time'].dt.year
dataAnomali['rank_from_all_station']    = dataAnomali.groupby(['year', 'month'])['anomali'].rank(ascending=False, method='min')
dataAnomali['rank_from_all_month']      = dataAnomali.groupby(['wmo_id', 'month'])['anomali'].rank(ascending=False, method='min')

# 10. Ringkasan Data Normal
dataNormal_Indo                         = dataNormal[['month','normal']].groupby('month').mean()
dataNormal_Indo['Jumlah_Stasiun_Valid']= dataNormal[['month','normal']].groupby('month').count()
dataNormal_Indo['Nama_Bulan']           = dataNormal['month'].apply(lambda x: calendar.month_name[x])
dataNormal_Indo['Total_Stasiun']        = dataNormal['wmo_id'].nunique()
dataNormal_Indo = dataNormal_Indo.reset_index(drop=False)
dataNormal_Indo['persentase_cakupan']     = (dataNormal_Indo['Jumlah_Stasiun_Valid'] / dataNormal_Indo['Total_Stasiun'] ) * 100


# 11. Anomali Indonesia Bulanan
dataAnomaliIndonesia                    = df_monthly[['time','value']].groupby('time').mean().reset_index()
dataAnomaliIndonesia['jumlah_stasiun_bulan_ini']  = df_monthly[['time','value']].groupby('time').count().reset_index(drop=True)
dataAnomaliIndonesia['year']            = dataAnomaliIndonesia['time'].dt.year
dataAnomaliIndonesia['month']           = dataAnomaliIndonesia['time'].dt.month
dataAnomaliIndonesia                    = dataAnomaliIndonesia.drop(columns='time')
dataAnomaliIND                          = pd.merge(dataAnomaliIndonesia, dataNormal_Indo, on='month').reset_index(drop=True)
dataAnomaliIND['anomali']               = dataAnomaliIND['value'] - dataAnomaliIND['normal']
dataAnomaliIND                          = dataAnomaliIND[['year','month','value','normal','anomali','Jumlah_Stasiun_Valid','jumlah_stasiun_bulan_ini']]
dataAnomaliIND                          = dataAnomaliIND.dropna(axis=0)
dataAnomaliIND['suhu_bulan_sebelum']    = dataAnomaliIND['value'].shift(1)
dataAnomaliIND['selisih_suhu']          = (dataAnomaliIND['value'] - dataAnomaliIND['suhu_bulan_sebelum'])
dataAnomaliIND['rank_from_all_month']   = dataAnomaliIND.groupby('month')['anomali'].rank(method='min', ascending=False).astype(int)
dataAnomaliIND['coverage_percentage']   = (dataAnomaliIND['jumlah_stasiun_bulan_ini'] / dataAnomaliIND['Jumlah_Stasiun_Valid'] * 100).round(2)
dataAnomaliIND['coverage_flag']         = np.where(dataAnomaliIND['coverage_percentage'] >= 80, 'VALID', 'LOW_COVERAGE')
dataAnomaliIND = dataAnomaliIND.rename(columns={'value':'trata'})
dataAnomali.to_csv(os.path.join(LONG_DIR, '06.TEMPERATURE_ANOMALI_DB.csv'), index=False)
dataAnomaliIND.to_csv(os.path.join(LONG_DIR, '06.TEMPERATURE_ANOMALI_INDONESIA_DB.csv'), index=False)
dataNormal_Indo = dataNormal_Indo.rename(columns={'normal':'Normal_Indonesia'})
dataNormal_Indo.to_csv(os.path.join(LONG_DIR, '06.TEMPERATURE_NORMAL_SUMMARY_DB.csv'), index=False)
print("Proses perhitungan anomali suhu rata-rata selesai.")